# Multi-model Comparison on Synthetic Data

Compare linear / proximity / density / ensemble detectors with ROC-AUC and F1.
Mirrors `compare_models.py`, but kept interactive for exploration.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from pyod.models.abod import ABOD
from pyod.models.cblof import CBLOF
from pyod.models.hbos import HBOS
from pyod.models.iforest import IForest
from pyod.models.knn import KNN
from pyod.models.lof import LOF
from pyod.models.mcd import MCD
from pyod.models.ocsvm import OCSVM
from pyod.models.pca import PCA
from pyod.utils.data import generate_data

from utils.metrics import evaluate_detector, summarize_results

In [ ]:
contamination = 0.15
random_state = 11
X, y = generate_data(
    behaviour='new',
    n_features=5,
    train_only=True,
    contamination=contamination,
    random_state=13,
)

classifiers = {
    'PCA': PCA(contamination=contamination, random_state=random_state),
    'MCD': MCD(contamination=contamination, random_state=random_state),
    'OCSVM': OCSVM(contamination=contamination),
    'KNN': KNN(contamination=contamination),
    'HBOS': HBOS(contamination=contamination),
    'LOF': LOF(n_neighbors=35, contamination=contamination),
    'CBLOF': CBLOF(contamination=contamination, check_estimator=False, random_state=random_state),
    'ABOD': ABOD(contamination=contamination),
    'IForest': IForest(contamination=contamination, random_state=random_state),
}

In [ ]:
results = {}
for name, clf in classifiers.items():
    clf.fit(X)
    results[name] = evaluate_detector(y, clf.labels_, clf.decision_scores_)

summarize_results(results, sort_by='roc_auc')
pd.DataFrame(results).T.sort_values('roc_auc', ascending=False)

## Reading the table

- **ROC-AUC / AP**: ranking quality of continuous scores (preferred for unsupervised AD)
- **Precision / Recall / F1**: depend on the contamination-based threshold

On well-separated synthetic blobs, tree ensembles and neighborhood methods usually lead;
on real ODDS data the ranking can change — see `03_odds_benchmark.ipynb`.